<div class='heading'>
    <div style='float:left;'><h1>CPSC 8810 Machine Learning for Graphs</h1></div>
     <img style="float: right; padding-right: 10px" width="100" src="https://raw.githubusercontent.com/bsethwalker/clemson-cs4300/main/images/clemson_paw.png"> </div>
     </div>

**Clemson University**<br>
**Fall 2025**<br>
**Instructor(s):** Aaron Masino <br>

## Homework 4: Graph Transformers and Heterogeneous Graphs
This homework is intended to assess your knowledge of core elements of graph transformers and heterogeneous graphs as introduced during the in-class lectures and labs. You may wish to refer to the course lectures and labs while completing this assignment.

**Unless otherwise noted in the problem instructions you may use any of the following Python libraries to complete the exercises:**
- numpy, scipy
- scikit-learn
- matplotlib, seaborn, pygraphviz
- PyTorch, PyTorch Geometric
- NetworkX

**Test code:** You may add test code after an exercise to evaluate your code. This is optional.


In [1]:
# Google Colab setup
# mount the google drive - this is necessary to access supporting src
from google.colab import drive
drive.mount("/content/drive")

# Create output directory
import os
from pathlib import Path

# data directory
DATA_DIR = Path('"/content/drive/MyDrive/Colab Notebooks/ml4g/data')
DATA_DIR.mkdir(parents=True, exist_ok=True)

Mounted at /content/drive


In [2]:
# install missing libraries (this may take several minutes)
!pip3 install torch_geometric

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.7/63.7 kB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 76.0 MB/s eta 0:00:00


In [3]:
import numpy as np
from pathlib import Path
import os
import numpy as np

# Graph analysis
import networkx as nx

# PyTorch and PyTorch Geometric
import torch
from torch.utils.data import TensorDataset
import torch.nn.functional as F
# import torch.nn as tnn so as not to conflict with nn from PyTorch Geometric
from torch import nn as tnn
from torch_geometric.datasets import FakeDataset
from torch_geometric.utils import to_networkx
from torch_geometric.data import Data, HeteroData
from torch_geometric.seed import seed_everything
from torch_geometric.transforms import BaseTransform
from torch_geometric.loader import DataLoader
from torch_geometric.nn import TransformerConv, global_mean_pool

import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.metrics import (classification_report, confusion_matrix, roc_curve, auc)
from sklearn.preprocessing import label_binarize

# set device
if torch.cuda.is_available():
    device = torch.device('cuda')
elif torch.backends.mps.is_available():
    device = torch.device('mps')
else:
    device = torch.device('cpu')

print(f"Using device: {device}")

RANDOM_SEED = 654321

Using device: cuda


---
# Part 1: Graph Transformers for Position Aware Tasks

## (4 points) Excercise 1: Anchor node selection

In this excerise, you are asked to implement a method that randomly selects a set of `k` anchor nodes from an input PyG graph, `data`. The method should **return a PyTorch Tensor** that contains the indices of the selected anchor nodes for the graph and is of shape `(N, k)` where `N` is the number of nodes. The sampling of anchor nodes should be done **without replacement**. If `k` is greater than the number of nodes in the graph, the method should raise a `ValueError`.

In [4]:
def select_anchors(data, k, seed=RANDOM_SEED):
    """Select k anchor nodes randomly from the input graph data object.

    Args:
        data (torch_geometric.data.Data): graph data object.
        k (int): Number of anchor nodes to select.
        seed (int): Random seed for reproducibility.

    Returns:
        torch.Tensor: Indices of the selected anchor nodes.
    """
    seed_everything(seed)

    ##################### YOUR CODE HERE #####################
    anchors = None

    return anchors

In [5]:
# check solution
seed_everything(RANDOM_SEED)
ds = FakeDataset(num_graphs=10, avg_num_nodes=50, num_channels=8, num_classes=2, task='graph')
anchors = select_anchors(ds[0], k=5, seed=RANDOM_SEED)
assert anchors.shape == (5,)
assert anchors.tolist()==[19, 1, 27, 38, 31]
try:
    select_anchors(ds[0], k=100, seed=RANDOM_SEED)
    assert False, "Expected ValueError for k greater than number of nodes"
except ValueError:
    pass
seed_everything(RANDOM_SEED)

## (5 points) Excercise 2: Distance to anchors positional embedding
In this exercise, you are asked to complete the `forward` method in the `AddAnchorDistancePE` class which is used as a Transform (not Transformer) on PyG graph data. We saw similar transforms, [AddRandomWalkPE](https://pytorch-geometric.readthedocs.io/en/latest/generated/torch_geometric.transforms.AddRandomWalkPE.html#torch_geometric.transforms.AddRandomWalkPE) and [AddLaplacianEigenvectorPE](https://pytorch-geometric.readthedocs.io/en/latest/generated/torch_geometric.transforms.AddLaplacianEigenvectorPE.html#torch_geometric.transforms.AddLaplacianEigenvectorPE), in the lab. Those transforms added postitional embeddings that represented graph structure. The `AddAnchorDistancePE` Transform will add positional embeddings that capture a node's position in the graph using distance to anchors.

To complete the `forward` method, you will need to use the class attributes `self.k` and `self.seed` and the input to the `forward` method `data` (which is a PyG graph data object) to select anchors for the graph using the `select_anchors` method you created above. You will need to compute the shortest distance between each node and each anchor **HINT:** convert the PyG graph, `data` to a Networkx graph and use the [nx.shortest_path_length](https://networkx.org/documentation/stable/reference/algorithms/generated/networkx.algorithms.shortest_paths.generic.shortest_path_length.html#networkx.algorithms.shortest_paths.generic.shortest_path_length) method. Use the shortest path distance from the node to the anchors to update the `distances` variable which **must** be a PyTorch tensor. The distances tensor should be added to the `data` object as the `pe` attribute after the anchor distances have been updated for each node.

In [6]:
class AddAnchorDistancePE(BaseTransform):
    def __init__(self, k, seed=RANDOM_SEED):
        self.k = k
        self.seed = seed

    def forward(self, data):
        distances = None

        ##################### YOUR CODE HERE #####################


        data.pe = distances
        return data

In [7]:
seed_everything(RANDOM_SEED)
ds = FakeDataset(num_graphs=1, avg_num_nodes=30, num_channels=8, num_classes=2, task='graph', transform=AddAnchorDistancePE(k=5, seed=RANDOM_SEED))
print(ds[0])
print(ds[0].pe.shape)
assert ds[0].pe.shape == (ds[0].num_nodes, 5)
assert ds[0].pe[0].tolist() == [2, 1, 1, 1, 1]

Data(y=[1], edge_index=[2, 316], x=[24, 8], pe=[24, 5])
torch.Size([24, 5])


---
# Part 2: Heterogeneous Graphs

## (3 points) Excercise 3: Maximum relation types in a heterogeneous graph
In this exercise, you are asked to complete the `max_relations` method that computes the maximum number of relations that are possible for a heterogeneous graph with _N_ node types and _E_ edge types. Recall that a relation, _R_ is defined by the triple _(NH, E, NT)_ where _NH_ is the node type for the head node, _E_ is the edge type, and _NT_ is the node type for the tail node.

In [8]:
def max_relations(num_node_types, num_edge_types):
    """Compute the maximum number of possible relation types in a heterogeneous graph.

    Args:
        num_node_types (int): Number of distinct node types, N.
        num_edge_types (int): Number of distinct edge types, E.

    Returns:
        int: Maximum number of possible relation types.
    """
    ##################### YOUR CODE HERE #####################
    pass

In [9]:
assert max_relations(3, 4) == 36
assert max_relations(5, 2) == 50

## (3 points) Excercise 4: RCGN learning parameters count
In this exercise, you are asked to complete the `rcgn_num_learning_params` method which computes the total number of learning parameters in a **single layer** of a relational graph convolution network (RCGN). Recall the equation for updating the embedding, $\mathbf{h}_v$ for node $v$ in a given layer **with no self loop** is:

$\mathbf{h}_v^{k+1} = \sigma \left( \left[\sum_{r\in R} \sum_{u\in N_r\left(v\right)} \frac{1}{|N_r\left(v\right)|} \mathbf{W}_r^{(k)} \mathbf{h}_u^{(k)} \right]  \right)$

and **with a self loop** is

$\mathbf{h}_v^{k+1} = \sigma \left( \left[\sum_{r\in R} \sum_{u\in N_r\left(v\right)} \frac{1}{|N_r\left(v\right)|} \mathbf{W}_r^{(k)} \mathbf{h}_u^{(k)} \right] + \mathbf{W}_0^{(k)} \mathbf{h}_v^{(k)} \right)$

Assume that the each matrix, $\mathbf{W}_r$ is of the same size `(input_dim, output_dim)`, and if a self-loop is used, that $\mathbf{W}_0$ is also of size `(input_dim, output_dim)`.

In [10]:
def rcgn_num_learning_params(num_relation_types, input_dim, output_dim, use_self_loop=True):
    """Compute the number of learning parameters in a single RGCN layer.

    Args:
        num_relation_types (int): Number of relation types, R.
        input_dim (int): Input embedding dimension, d_k.
        output_dim (int): Output embedding dimension, d_{k+1}.
        use_self_loop (bool): Whether to include self-loop parameters.

    Returns:
        int: Total number of learning parameters.
    """
    ##################### YOUR CODE HERE ####################
    pass

In [11]:
# test num_learning_params
assert rcgn_num_learning_params(4, 16, 32, use_self_loop=True) == 2560
assert rcgn_num_learning_params(3, 8, 64, use_self_loop=False) == 1536

# (5 points) Exercise 5: Reducing RGCN parameters with block matrices
In this exercise, you are asked to complete the `initialize_rcgn_block_matrices` method which *randomly* initializes the learning parameters for a **single layer** of a relational graph convolution network (RCGN) when **using a block diagonal matrix** for the model weight matrices $\mathbf{W}_r which are of shape $\left( d_{out}, d_{in} \right)$ where $d_{in}$ is the size of the input node embedding and $d_{out}$ is the size of the output node embedding. Recall that the block diagonal weight matrix approach replaces the complete weight matrices with a block diagonal matrix with $B$ blocks. For computational efficiency, the blocks should be stored separately so that the off-diagonal zeros are not stored or used in computations. Thus, the method implemetnation should return a PyTorch Tensor of shape $\left(R, B, B_{out}, B_{in}\right)$ where
* $R$ is the number of relations
* $B$ is the number of blocks
* Each block has shape $B_{out} \times B_{in}$

**HINT**: Your implementation should check if the `d_in` and `d_out` have zero remainder when divided by `B`.

In [12]:
def initialize_rcgn_block_matrices(R, d_in, d_out, B):
    """Initialize RGCN block diagonal weight matrices.

    Args:
        R (int): Number of relation types.
        d_in (int): Input node embedding dimension.
        d_out (int): Output node embedding dimension.
        B (int): Number of blocks.

    Returns:
        torch.Tensor: Initialized block diagonal weight matrices of shape (R, B, B_out, B_in).
    """

    ##################### YOUR CODE HERE #####################
    pass


In [13]:
seed_everything(RANDOM_SEED)
block_matrices = initialize_rcgn_block_matrices(R=3, d_in=16, d_out=32, B=4)
assert block_matrices.shape == (3, 4, 8, 4)
np.testing.assert_array_almost_equal(block_matrices[0,0,0,:].numpy(), np.array([  0.9803,  1.1420, -0.3219, -0.6069]), decimal=4)
try:
    initialize_rcgn_block_matrices(R=2, d_in=10, d_out=20, B=3)
    assert False, "Expected ValueError for d_in and d_out not divisible by B"
except ValueError:
    pass

## (5 points) Exercise 6: RCGN node embedding update
In this exercise, you are asked to complete the `rgcn_update` method which computes updated embeddings at a single relational graph convolution network (RCGN) layer for **all nodes** in the graph using the partial matrix formulation. The partial matrix formulation for updating the embeddings for all nodes **with no self loop** is given by

$\mathbf{H}^{k+1} = \sigma\left(\left[\sum_{r\in R} \mathbf{D}_r^{-1} \mathbf{A}_r \mathbf{H}^{(k)} \mathbf{W}_r^{(k)} \right]\right)$

and *with self-loops* is given by:

$\mathbf{H}^{k+1} = \sigma\left(\left[\sum_{r\in R} \mathbf{D}_r^{-1} \mathbf{A}_r \mathbf{H}^{(k)} \mathbf{W}_r^{(k)} \right] + \mathbf{W}_0^{(k)} \mathbf{H}^{(k)} \right) $

where
* $\mathbf{H}$ is an $N \times d$ matrix, $N$ is the number of nodes, and $d$ is the embedding dimension
* $\mathbf{W}_r$ is the weight matrix for relation type $r$
* $\mathbf{A}_r$ is the adjacency matrix for relation type $r$
* $\mathbf{D}_r^{-1}$ is the diagonal matrix where $\left(\mathbf{D}_r^{-1} \right)_{v,v}=\frac{1}{|N_r\left(v\right)|}$, i.e., one over the degree of node $v$ for relation $r$
* $\sigma$ is any non-linear activation function


In your method implementation, if `W0` is `None`, then a self-loop should **not** be used, otherwise if `W0` is not `None` a self-loop should be used.

In [14]:
def rgcn_update(Wr, H, Ar, nla, W0=None):
    """
    Perform a single R-GCN update step.

    Args:
        Wr (torch.Tensor): Weight matrices for each relation type shape (num_relations, input_dim, output_dim)
        W0 (torch.Tensor): Weight matrix for self-loops shape (input_dim, output_dim)
        H (torch.Tensor): Node feature matrix of shape (num_nodes, feature_dim).
        Ar (torch.Tensor): Adjacency matrices for each relation type, shape (num_relations, num_nodes, num_nodes).
        nla (method): non-linear activation function.

    Returns:
        torch.Tensor: Updated node feature matrix.
    """
    ##################### YOUR CODE HERE #####################
    pass

In [15]:
# test rgcn_update
num_nodes = 4
feature_dim = 3
num_relations = 2

# create random input data
seed_everything(RANDOM_SEED)
H = torch.randn((num_nodes, feature_dim))
Wr = torch.randn((num_relations, feature_dim, feature_dim))
Ar = torch.randint(0, 2, (num_relations, num_nodes, num_nodes)).float()
W0 = torch.randn((feature_dim, feature_dim))
nla = torch.relu
H_updated = rgcn_update(Wr, H, Ar, nla, W0=W0)
assert H_updated.shape == (num_nodes, feature_dim)
np.testing.assert_array_almost_equal(H_updated[-1].cpu().detach().numpy(), np.array([2.4803, 0.5368, 0.0000]), decimal=4)